# Transformer

In this lecture, we will discuss transformers which is a famous movie series....

![](https://www.comingsoon.net/wp-content/uploads/sites/3/2023/06/Watch-the-Transformers-Movies-Before-Rise-of-the-Beasts.jpg)


Actually, we want to discuss this picture:

![](https://miro.medium.com/v2/resize:fit:760/format:webp/1*2vyKzFlzIHfSmOU_lnQE4A.png)

## Introduction:

**What is a transformer?**

Transormer is a new simple network architecture based solely on attention mechanisms, dispensing with recurrence and convolutions entirely.

**What does Transformer do?**

It is designed to do sequence modeling, like recurrent neural network, LSTM. For example, text generation. 

**Is the transformer better than RNN and LSTM?**

For more details, please check this [page](https://medium.com/@mroko001/rnn-vs-lstm-vs-transformers-unraveling-the-secrets-of-sequential-data-processing-c4541c4b09f).

Advantages of RNNs:

- Sequential Dependencies: RNNs are well-suited for tasks where past information significantly impacts future predictions, such as language modeling and time series forecasting.
- Simple and Intuitive: The simplicity of RNNs makes them easy to understand and implement.

Limitations of RNNs:

- Vanishing and Exploding Gradients: RNNs often suffer from vanishing or exploding gradients, making it challenging to capture long-range dependencies.
- Difficulty with Parallelism: Due to their sequential nature, RNNs are not naturally parallelizable, leading to slower training times.


Advantages of LSTMs:
- Capturing Long-Term Dependencies: LSTMs are designed to capture long-range dependencies more effectively, mitigating the vanishing gradient problem.
- Reduced Forgetting: The forget gate in LSTMs allows the model to retain essential information over longer sequences.

Limitations of LSTMs:

- Complexity: LSTMs are more complex than basic RNNs, which can make them harder to train and require more computational resources.

Advantages of Transformers:
- Parallel Processing: Transformers process all inputs in parallel, making them significantly faster than RNNs and LSTMs, which are sequential in nature. 
- Capturing Long-Range Dependencies: Through self-attention mechanisms, transformers excel at capturing long-range dependencies, making them ideal for tasks like machine translation and document summarization.
- Interpretable Attention: Transformers offer interpretability through attention mechanisms, enabling insights into which parts of the input sequence are crucial for predictions.

Limitations of Transformers:
- Complexity: Transformers can be computationally expensive and require a large amount of data for training effectively.


**Chatgpt is based on transformer**


## Attention

Attention is

$$\mathrm{Attention}(Q,K,V) = \mathrm{softmax}(\frac{QK^T}{\sqrt{d_k}})V,$$

where $Q\in\mathbb{R}^{m\times d_k}$ is called **query**, $K\in\mathbb{R}^{m\times d_k}$ is called **key**, and $V\in\mathbb{R}^{m\times d_v}$ is called **value**.

What is the meaning of attention? Let's explain it by using database perspective:

Suppose we send a query into the database, some operations will find out which key in the database is the most similar to the query. Once the key is located, it will send out the value corresponding to that key as an output. In the figure, the operation finds that the Query is most similar to Key 5, and hence gives us the value 5 as output.

![](https://miro.medium.com/v2/resize:fit:1400/format:webp/1*2LtI2SALmTVGIK7Rqlv_gg.png)


The Attention mechanism is a neural architecture that mimics this process of retrieval.

1. The attention mechanism measures the similarity between the query q and each key-value k_i.
2. This similarity returns a weight for each key value.
3. Finally, it produces an output that is the weighted combination of all the values in our database.

## Positional encoding:

Since our model contains no recurrence and no convolution, in order for the model to make use of the order of the sequence, we must inject some information about the relative or absolute position of the tokens in the sequence.

Consider the sentence "I love math"

| Word | Pos | Positional encoding matrix |
|:--------:|:--------:|:--------:|
|  I   |  0  |  P_{0,0}, P_{0,1}, ..., P_{0,d}    |
|  Love   |  1   |  P_{1,0}, P_{1,1}, ..., P_{1,d}   |
|  Math   |  2   |  P_{2,0}, P_{2,1}, ..., P_{2,d}   |


To compute positional encoding, we use the following two functions:

$$ P(k,i) = \sin(k/n^{2*i/d}) $$

$$ P(k,i) = \cos(k/n^{(2*i+1)/d}) $$

# PyTorch

In [2]:
import torch
import torch.nn as nn
import torch.optim as optim
import torch.utils.data as data
import math
import copy

#### Multi-Head Attention

In [3]:
class MultiHeadAttention(nn.Module):
    def __init__(self, d_model, num_heads):
        super(MultiHeadAttention, self).__init__()
        assert d_model % num_heads == 0, "d_model must be divisible by num_heads"
        
        self.d_model = d_model
        self.num_heads = num_heads
        self.d_k = d_model // num_heads
        
        self.W_q = nn.Linear(d_model, d_model)
        self.W_k = nn.Linear(d_model, d_model)
        self.W_v = nn.Linear(d_model, d_model)
        self.W_o = nn.Linear(d_model, d_model)
        
    def scaled_dot_product_attention(self, Q, K, V, mask=None):
        attn_scores = torch.matmul(Q, K.transpose(-2, -1)) / math.sqrt(self.d_k)
        if mask is not None:
            attn_scores = attn_scores.masked_fill(mask == 0, -1e9)
        attn_probs = torch.softmax(attn_scores, dim=-1)
        output = torch.matmul(attn_probs, V)
        return output
        
    def split_heads(self, x):
        batch_size, seq_length, d_model = x.size()
        return x.view(batch_size, seq_length, self.num_heads, self.d_k).transpose(1, 2)
        
    def combine_heads(self, x):
        batch_size, _, seq_length, d_k = x.size()
        return x.transpose(1, 2).contiguous().view(batch_size, seq_length, self.d_model)
        
    def forward(self, Q, K, V, mask=None):
        Q = self.split_heads(self.W_q(Q))
        K = self.split_heads(self.W_k(K))
        V = self.split_heads(self.W_v(V))
        
        attn_output = self.scaled_dot_product_attention(Q, K, V, mask)
        output = self.W_o(self.combine_heads(attn_output))
        return output

#### Position-wise Feed-Forward Networks

In [4]:
class PositionWiseFeedForward(nn.Module):
    def __init__(self, d_model, d_ff):
        super(PositionWiseFeedForward, self).__init__()
        self.fc1 = nn.Linear(d_model, d_ff)
        self.fc2 = nn.Linear(d_ff, d_model)
        self.relu = nn.ReLU()

    def forward(self, x):
        return self.fc2(self.relu(self.fc1(x)))

#### Positional encoding

In [5]:
class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_seq_length):
        super(PositionalEncoding, self).__init__()
        
        pe = torch.zeros(max_seq_length, d_model)
        position = torch.arange(0, max_seq_length, dtype=torch.float).unsqueeze(1)
        div_term = torch.exp(torch.arange(0, d_model, 2).float() * -(math.log(10000.0) / d_model))
        
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        
        self.register_buffer('pe', pe.unsqueeze(0))
        
    def forward(self, x):
        return x + self.pe[:, :x.size(1)]

#### Encoder layer

In [6]:
class EncoderLayer(nn.Module):
    def __init__(self, d_model, num_heads, d_ff, dropout):
        super(EncoderLayer, self).__init__()
        self.self_attn = MultiHeadAttention(d_model, num_heads)
        self.feed_forward = PositionWiseFeedForward(d_model, d_ff)
        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)
        self.dropout = nn.Dropout(dropout)
        
    def forward(self, x, mask):
        attn_output = self.self_attn(x, x, x, mask)
        x = self.norm1(x + self.dropout(attn_output))
        ff_output = self.feed_forward(x)
        x = self.norm2(x + self.dropout(ff_output))
        return x

#### Decoder layer

In [7]:
class DecoderLayer(nn.Module):
    def __init__(self, d_model, num_heads, d_ff, dropout):
        super(DecoderLayer, self).__init__()
        self.self_attn = MultiHeadAttention(d_model, num_heads)
        self.cross_attn = MultiHeadAttention(d_model, num_heads)
        self.feed_forward = PositionWiseFeedForward(d_model, d_ff)
        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)
        self.norm3 = nn.LayerNorm(d_model)
        self.dropout = nn.Dropout(dropout)
        
    def forward(self, x, enc_output, src_mask, tgt_mask):
        attn_output = self.self_attn(x, x, x, tgt_mask)
        x = self.norm1(x + self.dropout(attn_output))
        attn_output = self.cross_attn(x, enc_output, enc_output, src_mask)
        x = self.norm2(x + self.dropout(attn_output))
        ff_output = self.feed_forward(x)
        x = self.norm3(x + self.dropout(ff_output))
        return x

#### Transformer

In [8]:
class Transformer(nn.Module):
    def __init__(self, src_vocab_size, tgt_vocab_size, d_model, num_heads, num_layers, d_ff, max_seq_length, dropout):
        super(Transformer, self).__init__()
        self.encoder_embedding = nn.Embedding(src_vocab_size, d_model)
        self.decoder_embedding = nn.Embedding(tgt_vocab_size, d_model)
        self.positional_encoding = PositionalEncoding(d_model, max_seq_length)

        self.encoder_layers = nn.ModuleList([EncoderLayer(d_model, num_heads, d_ff, dropout) for _ in range(num_layers)])
        self.decoder_layers = nn.ModuleList([DecoderLayer(d_model, num_heads, d_ff, dropout) for _ in range(num_layers)])

        self.fc = nn.Linear(d_model, tgt_vocab_size)
        self.dropout = nn.Dropout(dropout)

    def generate_mask(self, src, tgt):
        src_mask = (src != 0).unsqueeze(1).unsqueeze(2)
        tgt_mask = (tgt != 0).unsqueeze(1).unsqueeze(3)
        seq_length = tgt.size(1)
        nopeak_mask = (1 - torch.triu(torch.ones(1, seq_length, seq_length), diagonal=1)).bool()
        tgt_mask = tgt_mask & nopeak_mask
        return src_mask, tgt_mask

    def forward(self, src, tgt):
        src_mask, tgt_mask = self.generate_mask(src, tgt)
        src_embedded = self.dropout(self.positional_encoding(self.encoder_embedding(src)))
        tgt_embedded = self.dropout(self.positional_encoding(self.decoder_embedding(tgt)))

        enc_output = src_embedded
        for enc_layer in self.encoder_layers:
            enc_output = enc_layer(enc_output, src_mask)

        dec_output = tgt_embedded
        for dec_layer in self.decoder_layers:
            dec_output = dec_layer(dec_output, enc_output, src_mask, tgt_mask)

        output = self.fc(dec_output)
        return output

In [9]:
src_vocab_size = 5000
tgt_vocab_size = 5000
d_model = 512
num_heads = 8
num_layers = 6
d_ff = 2048
max_seq_length = 100
dropout = 0.1

transformer = Transformer(src_vocab_size, tgt_vocab_size, d_model, num_heads, num_layers, d_ff, max_seq_length, dropout)

# Generate random sample data
src_data = torch.randint(1, src_vocab_size, (64, max_seq_length))  # (batch_size, seq_length)
tgt_data = torch.randint(1, tgt_vocab_size, (64, max_seq_length))  # (batch_size, seq_length)

In [10]:
criterion = nn.CrossEntropyLoss(ignore_index=0)
optimizer = optim.Adam(transformer.parameters(), lr=0.0001, betas=(0.9, 0.98), eps=1e-9)

transformer.train()

for epoch in range(100):
    optimizer.zero_grad()
    output = transformer(src_data, tgt_data[:, :-1])
    loss = criterion(output.contiguous().view(-1, tgt_vocab_size), tgt_data[:, 1:].contiguous().view(-1))
    loss.backward()
    optimizer.step()
    print(f"Epoch: {epoch+1}, Loss: {loss.item()}")

Epoch: 1, Loss: 8.687183380126953
Epoch: 2, Loss: 8.549701690673828
Epoch: 3, Loss: 8.478403091430664


KeyboardInterrupt: 

## References:

1. transformers modules: https://pypi.org/project/transformers/#files

Code:

2. https://towardsdatascience.com/how-to-code-the-transformer-in-pytorch-24db27c8f9ec

3. https://www.datacamp.com/tutorial/building-a-transformer-with-py-torch

4. https://machinelearningmastery.com/implementing-the-transformer-encoder-from-scratch-in-tensorflow-and-keras/

Theory:

5. https://arxiv.org/abs/1706.03762

6. https://towardsdatascience.com/all-you-need-to-know-about-attention-and-transformers-in-depth-understanding-part-1-552f0b41d021

7. https://machinelearningmastery.com/the-transformer-attention-mechanism/